# Stage 1 — DreamBooth + LoRA per [V]

In [ ]:
import torch, sys, os

print(f'Python:           {sys.version[:6]}')
print(f'PyTorch:          {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    print(f'GPU:              {torch.cuda.get_device_name(0)}')
    print(f'VRAM:      {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise RuntimeError('GPU not available')

In [ ]:
!pip install -q \
    diffusers==0.27.2 \
    transformers==4.40.0 \
    accelerate==0.29.3 \
    peft==0.10.0 \
    wandb==0.16.6 \
    torchmetrics==1.3.2 \
    Pillow tqdm pyyaml

In [ ]:
#Patch 
import sys
import numpy as np

if not hasattr(np, "float_"):
    np.float_ = np.float64
if not hasattr(np, "complex_"):
    np.complex_ = np.complex128

print('Patching complete')

In [ ]:
# Login HuggingFace and Wandb 

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
import wandb

secrets = UserSecretsClient()

# HuggingFace
hf_token = secrets.get_secret('HF_TOKEN')
login(token=hf_token)
print('HuggingFace: OK')

# Wandb
wandb_token = secrets.get_secret('WANDB_API_KEY')
wandb.login(key=wandb_token)
print('Wandb: OK')

In [ ]:
import os

WORK = '/kaggle/working'

for d in ['checkpoints/stage1', 'checkpoints/stage2',
          'generated', 'splits', 'logs']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

REPO_URL = 'https://github.com/alecanc/Few-Shot-Personalization-of-a-Diffusion-Model-for-Industrial-Defect-Synthesis'
!git clone {REPO_URL} {WORK}/repo

os.chdir(f'{WORK}/repo')
print(f'Working directory: {os.getcwd()}')
print(os.listdir('.'))

In [ ]:
# Verify MVTec AD

import os
import tarfile
from pathlib import Path

input_dir = Path('/kaggle/input/datasets/alessiacancemi/mvtec-anomaly-detection')


output_dir = Path('/kaggle/working/defect-synthesis/data')
output_dir.mkdir(parents=True, exist_ok=True)

archives = ['bottle.tar.xz', 'leather.tar.xz', 'metal_nut.tar.xz']

for archive in archives:
    archive_path = input_dir / archive
    cat_name = archive.split('.')[0]
    target_dir = output_dir / cat_name
    
    if not target_dir.exists():
        with tarfile.open(archive_path, "r:xz") as tar:
            tar.extractall(path=output_dir)

MVTEC_extracted = output_dir

for cat in ['bottle', 'metal_nut', 'leather']:
    cat_dir = MVTEC_extracted / cat
    if not cat_dir.exists():
        print(f'Missing: {cat_dir}')
        continue
    

    n_good = len(list((cat_dir / 'train/good').glob('*.png')))
    
    defect_types = [d.name for d in (cat_dir / 'test').iterdir() if d.is_dir() and d.name != 'good']
    
    print(f'{cat}: {n_good} clean | defect types: {defect_types}')

In [ ]:
import yaml

with open('defect-synthesis/config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['paths']['mvtec_root']  = str(MVTEC_extracted)
cfg['paths']['output_root'] = WORK
cfg['paths']['checkpoints'] = f'{WORK}/checkpoints'
cfg['paths']['splits_dir']  = f'{WORK}/splits'
cfg['paths']['logs']        = f'{WORK}/logs'

with open('defect-synthesis/config.yaml', 'w') as f:
    yaml.dump(cfg, f)

print('config.yaml updated')
print(yaml.dump(cfg['paths'], default_flow_style=False))

In [ ]:
# Generate split dataset

CATEGORY = 'bottle'  # to change for every category

!python data/splits.py --config config.yaml

# Verify splits
import json
with open(f'{WORK}/splits/{CATEGORY}_split.json') as f:
    split = json.load(f)

print(f"Stage 1: {len(split['stage1'])} images")
for dtype, imgs in split['stage2'].items():
    print(f"Stage 2 [{dtype}]: {len(imgs)} train | {len(split['eval'][dtype])} eval")

In [ ]:
# Generate prior images for Stage 1 training

import huggingface_hub
if not hasattr(huggingface_hub, "cached_download"):
    huggingface_hub.cached_download = huggingface_hub.hf_hub_download

%run /kaggle/working/repo/data/generate_prior_images.py --config /kaggle/working/repo/config.yaml --category {CATEGORY}
!python data/generate_prior_images.py \
    --config config.yaml \
    --category {CATEGORY}

# Verify prior images
prior_dir = Path(f'{WORK}/checkpoints/stage1/{CATEGORY}/prior_images')
print(f' Prior images generated: {len(list(prior_dir.glob("*.png")))}')

In [ ]:
# To download prior images generated
!zip -r /kaggle/working/priorImagesGenerated.zip /kaggle/working/checkpoints

In [ ]:
!pip install -q "numpy<2.0.0" "opencv-python==4.9.0.80" "opencv-contrib-python==4.9.0.80" "wandb==0.16.6"
!pip install -q --upgrade transformers accelerate peft diffusers torchao

In [ ]:
# Training Stage 1


%cd /kaggle/working/repo

!python -u train/train_stage1.py --config config.yaml --category bottle

In [ ]:
final_dir = Path(f'{WORK}/checkpoints/stage1/{CATEGORY}/final')
print('File in\'adapter:')
for f in final_dir.iterdir():
    print(f'  {f.name} ({f.stat().st_size/1e6:.1f} MB)')

In [ ]:
import torch
from safetensors.torch import load_file


lora_weight_path = final_dir / 'pytorch_model.bin'
if not lora_weight_path.exists():
    lora_weight_path = final_dir / 'adapter_model.safetensors'
print(f'Loading LoRA weights from: {lora_weight_path}')

if lora_weight_path.suffix == '.safetensors':
    state_dict = load_file(lora_weight_path)
else:
    state_dict = torch.load(str(lora_weight_path), map_location='cpu')
clean_state_dict = {}
for k, v in state_dict.items():
    new_key = k
    if k.startswith("base_model.model."):
        new_key = k.replace("base_model.model.", "")
    clean_state_dict[new_key] = v

pipe.load_lora_weights(clean_state_dict)
print('LoRA weights loaded successfully')

In [ ]:
# Save checkpoint

import shutil

export_dir = f'{WORK}/stage1_lora_{CATEGORY}_final'
shutil.copytree(str(final_dir), export_dir, dirs_exist_ok=True)

print(f'Checkpoint ready to\'export: {export_dir}')
